In [65]:
import open_clip
import torch
from PIL import Image
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os
import json
import random
from PIL import Image
import torch.nn.functional as F
import os


In [29]:
device = torch.device('cuda')
BATCH_SIZE = 4
LEARNING_RATE = 0.001
WEIGHT_DECAY=0.1

In [6]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')


In [11]:
with open('PhotoBook/v2/gold-extracted.json', 'r') as file:
    data = json.load(file)
    img_dsrptn_tpls = []
    for key in data.keys():
        path = "PhotoBook/images/" + key
                        
        for subkey in data[key]:
            for dictionary_object in data[key][subkey]:
                dscrptn = dictionary_object["Message_Text"]
                img_dsrptn_tpls.append((path, dscrptn))

In [15]:
def split_list(big_list, train_ratio=0.8, test_ratio=0.1, dev_ratio=0.1):
    
    random.seed(42)
    random.shuffle(big_list)

    total_len = len(big_list)
    intg_train = int(total_len*train_ratio)
    intg_dev = int(total_len*dev_ratio)
    
    train_list = big_list[:intg_train]
    dev_list = big_list[intg_train:(intg_dev+intg_train)]
    test_list = big_list[(intg_train+intg_dev):]

    return train_list, dev_list, test_list

train_list, dev_list, test_list = split_list(img_dsrptn_tpls)

In [59]:
class clip_DS(Dataset):
    def __init__(self, main_list, preprocess, tokenizer):
        self.main_list = main_list
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.main_list)
        
    def __getitem__(self, idx):
        img, description = self.main_list[idx][0], self.main_list[idx][1]
        image = self.preprocess(Image.open(img))
        text = self.tokenizer(description).squeeze(0)  
        return image, text

In [60]:
train_raw_DS = clip_DS(train_list, preprocess, tokenizer)
dev_raw_DS = clip_DS(dev_list, preprocess, tokenizer)
test_raw_DS = clip_DS(test_list, preprocess, tokenizer)

In [88]:
train_dataloader = DataLoader(dataset=train_raw_DS,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
#                              collate_fn=collate_fn,
                             )

dev_dataloader = DataLoader(dataset=dev_raw_DS,
                            batch_size=BATCH_SIZE,
#                            collate_fn=collate_fn,
                           )

test_dataloader = DataLoader(dataset=test_raw_DS,
                            batch_size=16,
#                             collate_fn=collate_fn,
                            )


In [62]:
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)

In [1]:
def contrastive_loss(logits):
    B = logits.size(0)
    labels = torch.arange(B, device=logits.device)
    loss_t2i = F.cross_entropy(logits, labels)
    loss_i2t = F.cross_entropy(logits.T, labels)
    return (loss_t2i + loss_i2t) / 2

In [70]:
def train(model, dataloader, val_dataloader, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_pbar = tqdm(dataloader, desc=f"Train Epoch {epoch+1}")
        
        for images, texts in train_pbar:
            images, texts = images.to(device), texts.to(device)
            
            image_features = F.normalize(model.encode_image(images), dim=-1)
            text_features = F.normalize(model.encode_text(texts), dim=-1)
            
            logits = image_features @ text_features.T /0.07
            loss = contrastive_loss(logits)

            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(dataloader)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            val_pbar = tqdm(val_dataloader, desc=f"Val Epoch {epoch+1}")
            for images, texts in val_pbar:
                images, texts = images.to(device), texts.to(device)
                
                image_features = F.normalize(model.encode_image(images), dim=-1)
                text_features = F.normalize(model.encode_text(texts), dim=-1)

                
                logits = image_features @ text_features.T /0.07
                loss = contrastive_loss(logits)
                
                val_loss += loss.item()
                val_pbar.set_postfix({'batch_loss': f"{loss.item():.4f}"})
        
        avg_val_loss = val_loss / len(val_dataloader)
        
        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")
    


In [72]:
train(model, train_dataloader, test_dataloader, num_epochs=4)

Val Epoch 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:01<00:00, 16.12it/s, batch_loss=1.0986]


Epoch 1: Train Loss=1.3852, Val Loss=1.3752


Val Epoch 2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:01<00:00, 15.59it/s, batch_loss=1.0986]


Epoch 2: Train Loss=1.3850, Val Loss=1.3753


Val Epoch 3: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:01<00:00, 15.84it/s, batch_loss=1.0986]


Epoch 3: Train Loss=1.3850, Val Loss=1.3752


Val Epoch 4: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:01<00:00, 16.02it/s, batch_loss=1.0985]

Epoch 4: Train Loss=1.3849, Val Loss=1.3752


In [83]:
def test_model(model, test_loader, device, top=1):
    model.eval()
    total = 0
    correct = 0
    
    with torch.no_grad():
        for images, texts in test_loader:
            images = images.to(device)
            texts = texts.to(device)
            
            image_features = F.normalize(model.encode_image(images), dim=-1)
            text_features = F.normalize(model.encode_text(texts), dim=-1)
            logits = 100.0 * image_features @ text_features.T  # [B, B]
            
            batch_size = logits.size(0)
            
            if top == 1:
                preds = logits.argmax(dim=1)
                labels = torch.arange(batch_size, device=device)
                correct += (preds == labels).sum().item()
                
            elif top == 5:
                _, top5_indices = logits.topk(5, dim=1)
                labels = torch.arange(batch_size, device=device).unsqueeze(1)
                correct += (top5_indices == labels).any(dim=1).sum().item()
            
            total += batch_size
    
    accuracy = correct / total
    print(f"Top-{top} Accuracy: {accuracy:.4f}")
    return accuracy

In [89]:
test_model(model, test_dataloader, device, top=5)

Top-5 Accuracy: 0.3495 (36/103)


0.34951456310679613